In [11]:
import numpy as np
import os
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from sklearn.model_selection import train_test_split
import json
from datetime import datetime
df = pd.read_csv("./messidor_data.csv")  


In [12]:
USE_VAL = False
BATCH_SIZE = 5
NUM_CLASSES = 2
IMG_SIZE = (224, 224)

ROTATION_RANGE = 0
WIDTH_SHIFT_RANGE = 0.0
HEIGHT_SHIFT_RANGE = 0.0
SHEAR_RANGE = 0.0
ZOOM_RANGE = 0.0
HORIZONTAL_FLIP = False
VERTICAL_FLIP = False

USE_AUG = True
if USE_AUG:
    ROTATION_RANGE = 40
    WIDTH_SHIFT_RANGE = 0.2
    HEIGHT_SHIFT_RANGE = 0.2
    SHEAR_RANGE = 0.2
    ZOOM_RANGE = 0.2
    HORIZONTAL_FLIP = True
    VERTICAL_FLIP = True

EPOCHS = 15

In [13]:
df['file_path'] = df['id_code'].apply(lambda x: 'images/' + x)
paths = df['file_path'].to_list()
labels = pd.get_dummies(df['adjudicated_dme'], dtype=float).to_numpy()

In [14]:
x_train, x_test, y_train, y_test = train_test_split(paths, labels, test_size=0.1, random_state=0, stratify=labels)
if USE_VAL == True:
    x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.1, random_state=0, stratify=y_train)

In [15]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=ROTATION_RANGE,
    width_shift_range=WIDTH_SHIFT_RANGE,
    height_shift_range=HEIGHT_SHIFT_RANGE,
    shear_range=SHEAR_RANGE,
    zoom_range=ZOOM_RANGE,
    horizontal_flip=HORIZONTAL_FLIP,
    vertical_flip=VERTICAL_FLIP,
    fill_mode='nearest'
)
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
def create_generator(file_paths, labels, datagen):
    images = []
    for f in file_paths:
        img = tf.keras.utils.load_img(f, target_size=IMG_SIZE)
        img_array = tf.keras.utils.img_to_array(img)
        images.append(img_array)
    images = np.array(images)
    labels = np.array(labels)
    return datagen.flow(images, labels, batch_size=BATCH_SIZE, shuffle=True)

train_gen = create_generator(x_train, y_train, train_datagen)
test_gen = create_generator(x_test, y_test, test_datagen)

steps_train = len(x_train) // BATCH_SIZE
steps_test = len(x_test) // BATCH_SIZE

if USE_VAL == True:
    val_gen = create_generator(x_val, y_val, val_datagen)
    steps_val = len(x_val) // BATCH_SIZE

In [17]:
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(NUM_CLASSES, activation='softmax')(x)

In [18]:
model = Model(inputs=base_model.input, outputs=predictions)

for layer in base_model.layers:
    layer.trainable = False

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(
    train_gen,
    validation_data= val_gen if USE_VAL == True else None,
    epochs = EPOCHS
)
loss, acc = model.evaluate(test_gen, steps=steps_test)
print(f"accuracy: {acc:.4f}")

Epoch 1/15


/opt/anaconda3/envs/dr_paper/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


314/314 ━━━━━━━━━━━━━━━━━━━━ 22s 54ms/step - accuracy: 0.9133 - loss: 0.3403
Epoch 2/15
314/314 ━━━━━━━━━━━━━━━━━━━━ 19s 59ms/step - accuracy: 0.9133 - loss: 0.3171
Epoch 3/15
314/314 ━━━━━━━━━━━━━━━━━━━━ 21s 67ms/step - accuracy: 0.9133 - loss: 0.3041
Epoch 4/15
314/314 ━━━━━━━━━━━━━━━━━━━━ 19s 62ms/step - accuracy: 0.9133 - loss: 0.3050
Epoch 5/15
314/314 ━━━━━━━━━━━━━━━━━━━━ 17s 54ms/step - accuracy: 0.9133 - loss: 0.3073
Epoch 6/15
314/314 ━━━━━━━━━━━━━━━━━━━━ 17s 54ms/step - accuracy: 0.9133 - loss: 0.3007
Epoch 7/15
314/314 ━━━━━━━━━━━━━━━━━━━━ 16s 51ms/step - accuracy: 0.9133 - loss: 0.2998
Epoch 8/15
314/314 ━━━━━━━━━━━━━━━━━━━━ 16s 51ms/step - accuracy: 0.9133 - loss: 0.3003
Epoch 9/15
314/314 ━━━━━━━━━━━━━━━━━━━━ 15s 49ms/step - accuracy: 0.9133 - loss: 0.3025
Epoch 10/15
314/314 ━━━━━━━━━━━━━━━━━━━━ 15s 48ms/step - accuracy: 0.9133 - loss: 0.3030
Epoch 11/15
314/314 ━━━━━━━━━━━━━━━━━━━━ 16s 50ms/step - accuracy: 0.9133 - loss: 0.2996
Epoch 12/15
314/314 ━━━━━━━━━━━━━━━━━━━━ 

In [19]:
import json
import pandas as pd
from datetime import datetime

with open('results.json', "r") as f:
    data = json.load(f)

data.append({
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "accuracy": f"{acc:.4f}",
    "batch_size": BATCH_SIZE,
    "img_size": IMG_SIZE,
    "epochs": EPOCHS,
    "num_classes": NUM_CLASSES,
    "augmentation": {
        "rotation_range": ROTATION_RANGE,
        "width_shift_range": WIDTH_SHIFT_RANGE,
        "height_shift_range": HEIGHT_SHIFT_RANGE,
        "shear_range": SHEAR_RANGE,
        "zoom_range": ZOOM_RANGE,
        "horizontal_flip": HORIZONTAL_FLIP,
        "vertical_flip": VERTICAL_FLIP,
        "rescale": 1./255
    }
})

with open('results.json', "w") as f:
    json.dump(data, f, indent=4)

df_results = pd.json_normalize(
    data, 
    sep='_'
)
df_results.to_csv('results.csv', index=False)